<a href="https://colab.research.google.com/github/kamal-gavel/Block-Matrix-Based-Financial-Sentiment-and-Market-Feature-Fusion-for-Portfolio-Optimization/blob/main/Block_Matrix_Based_Financial_Sentiment_and_Market_Feature_Fusion_for_Multi_Asset_Portfolio_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 🚀 BLOCK MATRIX + ML + PORTFOLIO (FINAL REALISTIC VERSION)
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from xgboost import XGBClassifier

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv("panel_dataset_extended_horizons.csv")

print("Original Shape:", df.shape)

# ============================================================
# 2. DATE PROCESSING
# ============================================================
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['company', 'date'])

# ============================================================
# 3. FEATURE GROUPING (BLOCK MATRIX IDEA)
# ============================================================
target = 'return_t+7'

sentiment_cols = [c for c in df.columns if 'sent' in c or 'news' in c or 'ratio' in c]
market_cols = [c for c in df.columns if 'return' in c or 'volatility' in c or 'trend' in c]

# Remove leakage
future_cols = ['return_t+7', 'return_t+9', 'return_t+10']
market_cols = [c for c in market_cols if c not in future_cols]

print("Sentiment Features:", len(sentiment_cols))
print("Market Features:", len(market_cols))

# ============================================================
# 4. TARGET
# ============================================================
df['target_up'] = (df[target] > 0).astype(int)

companies = df['company'].unique()

all_preds = []

# ============================================================
# 5. MODEL PER STOCK (BLOCK MATRIX PCA)
# ============================================================
for stock in companies:

    df_stock = df[df['company'] == stock].copy()

    if len(df_stock) < 50:
        continue  # skip very small series

    X_sent = df_stock[sentiment_cols]
    X_market = df_stock[market_cols]
    y = df_stock['target_up']

    # Standardization
    scaler_s = StandardScaler()
    scaler_m = StandardScaler()

    Xs = scaler_s.fit_transform(X_sent)
    Xm = scaler_m.fit_transform(X_market)

    # PCA (BLOCK MATRIX REDUCTION)
    pca_s = PCA(n_components=0.95)
    pca_m = PCA(n_components=0.95)

    Zs = pca_s.fit_transform(Xs)
    Zm = pca_m.fit_transform(Xm)

    # 🔥 BLOCK MATRIX CONCATENATION
    X_block = np.concatenate([Zs, Zm], axis=1)

    # Time split
    split = int(len(X_block) * 0.8)

    X_train, X_test = X_block[:split], X_block[split:]
    y_train, y_test = y[:split], y[split:]

    dates_test = df_stock['date'].iloc[split:]
    returns = df_stock[target].iloc[split:]

    # Model
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:,1]

    temp = pd.DataFrame({
        'date': dates_test.values,
        'company': stock,
        'proba': proba,
        'return': returns.values
    })

    all_preds.append(temp)

# ============================================================
# 6. COMBINE DATA (NO FORCED ALIGNMENT)
# ============================================================
df_all = pd.concat(all_preds)

# Pivot (allow missing values naturally)
pred_matrix = df_all.pivot(index='date', columns='company', values='proba')
return_matrix = df_all.pivot(index='date', columns='company', values='return')

# Fill missing safely
pred_matrix = pred_matrix.fillna(0.5)
return_matrix = return_matrix.fillna(0)

pred_matrix = pred_matrix.values
return_matrix = return_matrix.values

print("Matrix Shape:", pred_matrix.shape)

# ============================================================
# 7. PORTFOLIO (REALISTIC)
# ============================================================
def compute_weights(proba):
    w = proba - 0.5
    w = np.clip(w, -0.2, 0.2)
    if np.sum(np.abs(w)) > 0:
        w = w / np.sum(np.abs(w))
    return w

portfolio_returns = []
transaction_cost = 0.001

for t in range(len(pred_matrix)):

    proba = pred_matrix[t]

    # Confidence filter
    mask = np.abs(proba - 0.5) > 0.05

    if np.sum(mask) == 0:
        portfolio_returns.append(0)
        continue

    weights = compute_weights(proba * mask)

    r = np.dot(weights, return_matrix[t])

    # Transaction cost
    r -= transaction_cost * np.sum(np.abs(weights))

    portfolio_returns.append(r)

portfolio_returns = np.array(portfolio_returns)

# ============================================================
# 8. PERFORMANCE METRICS (ROBUST)
# ============================================================
total_return = np.sum(portfolio_returns)

mean_ret = np.mean(portfolio_returns)
std_ret = np.std(portfolio_returns)

if std_ret < 1e-4:
    sharpe = 0
else:
    sharpe = mean_ret / std_ret * np.sqrt(252)

# Drawdown
cum_returns = np.cumsum(portfolio_returns)
peak = np.maximum.accumulate(cum_returns)
drawdown = cum_returns - peak
max_dd = np.min(drawdown)

# ============================================================
# 9. CLASSIFICATION METRICS
# ============================================================
y_true = (return_matrix.flatten() > 0).astype(int)
y_pred = (pred_matrix.flatten() > 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# ============================================================
# 10. RESULTS
# ============================================================
print("\n===================================")
print("📊 MODEL PERFORMANCE")
print("===================================")
print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall   :", round(rec,4))
print("F1 Score :", round(f1,4))

print("\n===================================")
print("📈 PORTFOLIO PERFORMANCE")
print("===================================")
print("Total Return :", round(total_return,4))
print("Sharpe Ratio :", round(sharpe,4))
print("Max Drawdown :", round(max_dd,4))

# ============================================================
# 11. SAVE
# ============================================================
pd.DataFrame({
    "portfolio_returns": portfolio_returns
}).to_csv("final_github_portfolio.csv", index=False)

print("\n✅ Saved: final_github_portfolio.csv")
print("\n🚀 FINAL GITHUB-READY PIPELINE COMPLETE")

Original Shape: (1521, 34)
Sentiment Features: 18
Market Features: 8
Matrix Shape: (213, 5)

📊 MODEL PERFORMANCE
Accuracy : 0.8986
Precision: 0.6491
Recall   : 0.8409
F1 Score : 0.7327

📈 PORTFOLIO PERFORMANCE
Total Return : 1.242
Sharpe Ratio : 5.0022
Max Drawdown : -0.1594

✅ Saved: final_github_portfolio.csv

🚀 FINAL GITHUB-READY PIPELINE COMPLETE


In [2]:
# ============================================================
# 🚀 FINAL LEAKAGE-FREE BLOCK MATRIX + PORTFOLIO PIPELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from xgboost import XGBClassifier

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv("panel_dataset_extended_horizons.csv")

print("Original Shape:", df.shape)

# ============================================================
# 2. DATE PROCESSING
# ============================================================
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['company', 'date'])

# ============================================================
# 3. FEATURE GROUPING (BLOCK MATRIX)
# ============================================================
target = 'return_t+7'

sentiment_cols = [c for c in df.columns if 'sent' in c or 'news' in c or 'ratio' in c]
market_cols = [c for c in df.columns if 'return' in c or 'volatility' in c or 'trend' in c]

# 🚨 REMOVE ALL LEAKAGE FEATURES
leakage_cols = [
    'return_t+7','return_t+9','return_t+10',
    'return_1','return_3','return_5'
]

market_cols = [c for c in market_cols if c not in leakage_cols]

print("Sentiment Features:", len(sentiment_cols))
print("Market Features:", len(market_cols))

# ============================================================
# 4. TARGET
# ============================================================
df['target_up'] = (df[target] > 0).astype(int)

companies = df['company'].unique()

all_preds = []

# ============================================================
# 5. MODEL PER STOCK (BLOCK MATRIX PCA)
# ============================================================
for stock in companies:

    df_stock = df[df['company'] == stock].copy()

    if len(df_stock) < 60:
        continue

    X_sent = df_stock[sentiment_cols]
    X_market = df_stock[market_cols]
    y = df_stock['target_up']

    # Standardization
    scaler_s = StandardScaler()
    scaler_m = StandardScaler()

    Xs = scaler_s.fit_transform(X_sent)
    Xm = scaler_m.fit_transform(X_market)

    # PCA per block
    pca_s = PCA(n_components=0.95)
    pca_m = PCA(n_components=0.95)

    Zs = pca_s.fit_transform(Xs)
    Zm = pca_m.fit_transform(Xm)

    # 🔥 BLOCK MATRIX
    X_block = np.concatenate([Zs, Zm], axis=1)

    # ========================================================
    # 🚨 TIME SPLIT WITH GAP (VERY IMPORTANT)
    # ========================================================
    split = int(len(X_block) * 0.8)
    gap = 5

    X_train = X_block[:split-gap]
    X_test = X_block[split:]

    y_train = y[:split-gap]
    y_test = y[split:]

    dates_test = df_stock['date'].iloc[split:]
    returns = df_stock[target].iloc[split:]

    # Handle imbalance
    scale_pos_weight = (len(y_train) - sum(y_train)) / (sum(y_train) + 1e-9)

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:,1]

    temp = pd.DataFrame({
        'date': dates_test.values,
        'company': stock,
        'proba': proba,
        'return': returns.values
    })

    all_preds.append(temp)

# ============================================================
# 6. COMBINE DATA
# ============================================================
df_all = pd.concat(all_preds)

pred_matrix = df_all.pivot(index='date', columns='company', values='proba')
return_matrix = df_all.pivot(index='date', columns='company', values='return')

# No forced alignment
pred_matrix = pred_matrix.fillna(0.5)
return_matrix = return_matrix.fillna(0)

pred_matrix = pred_matrix.values
return_matrix = return_matrix.values

print("Matrix Shape:", pred_matrix.shape)

# ============================================================
# 7. PORTFOLIO (REALISTIC)
# ============================================================
def compute_weights(proba):
    w = proba - 0.5
    w = np.clip(w, -0.2, 0.2)
    if np.sum(np.abs(w)) > 0:
        w = w / np.sum(np.abs(w))
    return w

portfolio_returns = []
transaction_cost = 0.001

for t in range(len(pred_matrix)):

    proba = pred_matrix[t]

    # Confidence filter
    mask = np.abs(proba - 0.5) > 0.05

    if np.sum(mask) == 0:
        portfolio_returns.append(0)
        continue

    weights = compute_weights(proba * mask)

    r = np.dot(weights, return_matrix[t])

    r -= transaction_cost * np.sum(np.abs(weights))

    portfolio_returns.append(r)

portfolio_returns = np.array(portfolio_returns)

# ============================================================
# 8. PERFORMANCE (ROBUST)
# ============================================================
total_return = np.sum(portfolio_returns)

mean_ret = np.mean(portfolio_returns)
std_ret = np.std(portfolio_returns)

if std_ret < 1e-4:
    sharpe = 0
else:
    sharpe = mean_ret / std_ret * np.sqrt(252)

# Drawdown
cum_returns = np.cumsum(portfolio_returns)
peak = np.maximum.accumulate(cum_returns)
drawdown = cum_returns - peak
max_dd = np.min(drawdown)

# ============================================================
# 9. MODEL METRICS
# ============================================================
y_true = (return_matrix.flatten() > 0).astype(int)
y_pred = (pred_matrix.flatten() > 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Baseline
majority_acc = max(np.mean(y_true), 1-np.mean(y_true))

# ============================================================
# 10. RESULTS
# ============================================================
print("\n===================================")
print("📊 FINAL MODEL PERFORMANCE")
print("===================================")
print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall   :", round(rec,4))
print("F1 Score :", round(f1,4))
print("Baseline :", round(majority_acc,4))

print("\n===================================")
print("📈 FINAL PORTFOLIO PERFORMANCE")
print("===================================")
print("Total Return :", round(total_return,4))
print("Sharpe Ratio :", round(sharpe,4))
print("Max Drawdown :", round(max_dd,4))

# ============================================================
# 11. SAVE
# ============================================================
pd.DataFrame({
    "portfolio_returns": portfolio_returns
}).to_csv("final_publishable_portfolio.csv", index=False)

print("\n✅ Saved: final_publishable_portfolio.csv")
print("\n🚀 FINAL LEAKAGE-FREE PIPELINE COMPLETE")

Original Shape: (1521, 34)
Sentiment Features: 18
Market Features: 6
Matrix Shape: (213, 5)

📊 FINAL MODEL PERFORMANCE
Accuracy : 0.9108
Precision: 0.7015
Recall   : 0.8011
F1 Score : 0.748
Baseline : 0.8347

📈 FINAL PORTFOLIO PERFORMANCE
Total Return : 1.5142
Sharpe Ratio : 6.1503
Max Drawdown : -0.115

✅ Saved: final_publishable_portfolio.csv

🚀 FINAL LEAKAGE-FREE PIPELINE COMPLETE
